## Script para recopilar datos de tipo secundarios.

En este script se implementa una estrategia de recopilación de datos de tipo secundarios basada en Web Scraping.

Web Scraping es una técnica automatizada utilizada para extraer grandes cantidades de datos no estructurados de sitios web diversos y convertirlos en información estructurada.

### Fase 1 Recopilación de datos.

En esta parte del script se implementa la lógica para acceder a la web y descargarse los datos no estructurados. Para ello, vamos a utilizar la librería BeautifulSoup (https://beautiful-soup-4.readthedocs.io/en/latest/) que nos permite analizar fácilmente el contenido web.

Lo primero que vamos a hacer es instalarnos la librería para poder utilizarla en nuestro script.

In [1]:
!pip install beautifulsoup4

In [2]:
from bs4 import BeautifulSoup # importamos la librería instalada -> nos permite acceder al contenido de la web
from urllib.request import urlopen # nos abrir una conexión con la página web a descargar.

url = 'https://www.rottentomatoes.com/browse/movies_in_theaters/sort:popular'     # url de la página a analizar su contenido
page = urlopen(url)                         # nos conectamos a la web.
html = page.read().decode('utf-8')          # guardamos el contenido en una variable
soup = BeautifulSoup(html, 'html.parser')   # utlizamos BeautifulSoup para recorrer su estructura.

films_rs = soup.find_all('span',class_='p--small')         # vamos a extraer el listado de películas.
data_films = list()
for film in films_rs:
  parent = film.parent
  critic = parent.find('rt-text',{'slot':'criticsScore'}) #obtenemos su critica de crítico
  audience = parent.find('rt-text',{'slot':'audienceScore'}) #obtenemos su critica de la audencia
  if(film is not None and critic is not None and audience is not None):
    #antes de guardar los datos, accedemos a su resumen
    url_resume = parent.parent.find('a',{'data-qa':'discovery-media-list-item-caption'})
    #construimos la url de la página a la que queremos acceder
    url_resume = 'https://www.rottentomatoes.com/'+(url_resume['href'])
    page = urlopen(url_resume)
    html2 = page.read().decode('utf-8')
    soup2 = BeautifulSoup(html2, 'html.parser')
    resume = soup2.find_all('rt-text',{'slot':'content'})  #recuperamos su resumen
    if(resume is not None):
      #guardamos su resumen
      data_films.append({'title':film.text,
                        'critic':critic.text,
                        'audience':audience.text,
                        'resume':resume[0].text})
  else:
    print('not found')    # pelicula no encontrada

print(f'size:{len(data_films)}')
print(f'ID\tTitulo\tCritico\tAudiencia\tResumen')
# utilizamos un bucle para mostrar toda la información almacenada.
for idx,item in enumerate(data_films):

  print(f'{idx}\t{item['title']}\t{item['critic']}\t{item['audience']}\t{item['resume']}\n')

not found
not found
not found
not found
size:28
ID	Titulo	Critico	Audiencia	Resumen
0	
          Wuthering Heights
        	 58%	 77%	
                    Tragedy strikes when Heathcliff falls in love with Catherine Earnshaw, a woman from a wealthy family in 18th-century England.
                

1	
          Psycho Killer
        	 10%	 37%	
                    Following the brutal murder of her husband, a Kansas highway patrol officer (Georgina Campbell) sets out on a journey to track down the perpetrator. As the hunt progresses, she comes to realize the man responsible (James Preston Rogers) is a sadistic serial killer, and the depth of his mental depravity and his sinister agenda is more twisted than anyone could have imagined.
                

2	
          Crime 101
        	 88%	 85%	
                    Set against the sun-bleached grit of Los Angeles, Crime 101 weaves the tale of an elusive jewel thief (Chris Hemsworth) whose string of heists along the 101 freeway have mystif

### Fase 2: Estructuración de datos.

En en esta fase del script se pasa de datos no estructurados a estructurados, dándole cierta coherencia y estructura a los datos extraídos de la web. Se utilizará la librería JSON para guardar los datos extraídos en un formato que sea fácilmente trabajar con ellos.

Para poder utilizar la librería, tenemos que importarla en nuestro script. No hace falta instalarla porque ya viene incorporada.


In [ ]:
import json

# convertimos la información almacenada en json.
json_str = json.dumps(data_films)
# displaying
print(type(json_str))
print("Datos en formato json: \n")
print(json_str)

<class 'str'>
Datos en formato json: 

[{"title": "\n          Wuthering Heights\n        ", "critic": " 59%", "audience": " 77%", "resume": "\n                    Tragedy strikes when Heathcliff falls in love with Catherine Earnshaw, a woman from a wealthy family in 18th-century England.\n                "}, {"title": "\n          Crime 101\n        ", "critic": " 88%", "audience": " 85%", "resume": "\n                    Set against the sun-bleached grit of Los Angeles, Crime 101 weaves the tale of an elusive jewel thief (Chris Hemsworth) whose string of heists along the 101 freeway have mystified police. When he eyes the score of a lifetime, his path crosses that of a disillusioned insurance broker (Halle Berry) who is facing her own crossroads. Convinced he has found a pattern, a relentless detective (Mark Ruffalo) is closing in, raising the stakes even higher. As the heist approaches, the line between hunter and hunted begins to blur, and all three are faced with life-defining cho